In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!pip install -q transformers accelerate pillow
import torch, transformers
print("GPU:", torch.cuda.get_device_name(0), "| transformers:", transformers.__version__)

GPU: Tesla T4 | transformers: 5.0.0


In [3]:
import glob, pandas as pd
from pathlib import Path

RUN = pd.read_csv(glob.glob("/kaggle/input/**/run_list.csv", recursive=True)[0])
IMG = {}
for p in glob.glob("/kaggle/input/**/batch1-*.jpg", recursive=True):
    if "batch_3" in p:
        continue
    IMG.setdefault(Path(p).stem, p)
RUN["path"] = RUN["file_name"].map(IMG)
RUN = RUN.dropna(subset=["path"]).reset_index(drop=True)
print(f"실행 목록 {len(RUN):,}건 / 이미지 {len(IMG):,}장")

실행 목록 1,413건 / 이미지 1,489장


In [4]:
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch

MODEL  = "PaddlePaddle/PaddleOCR-VL-1.6"
PROMPT = "Table Recognition:"

proc  = AutoProcessor.from_pretrained(MODEL)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL, dtype=torch.bfloat16).to("cuda").eval()
print("로드 완료 |", next(model.parameters()).dtype, "|", PROMPT)

processor_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/641 [00:00<?, ?B/s]

The image processor of type `PaddleOCRVLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

...

Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'mrope_section'}


model.safetensors:   0%|          | 0.00/1.92G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/608 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/133 [00:00<?, ?B/s]

로드 완료 | torch.bfloat16 | Table Recognition:


In [5]:
from PIL import Image
import torch

def parse(path, max_tokens=3000):
    img = Image.open(path).convert("RGB")
    msgs = [{"role": "user", "content": [
        {"type": "image", "image": img},
        {"type": "text", "text": PROMPT}]}]
    inp = proc.apply_chat_template(msgs, add_generation_prompt=True,
                                   tokenize=True, return_dict=True,
                                   return_tensors="pt").to("cuda")
    with torch.no_grad():
        o = model.generate(**inp, max_new_tokens=max_tokens,
                           do_sample=False, use_cache=True)
    return proc.decode(o[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)

In [6]:
import json, time
from pathlib import Path

OUT, END = Path("/kaggle/working/paddle_raw.jsonl"), 150
t0, n_ok, n_err = time.time(), 0, 0

with OUT.open("w", encoding="utf-8") as f:
    for _, row in RUN.iloc[:END].iterrows():
        t = time.time()
        try:
            rec = {"file_name": row["file_name"], "seq": int(row["seq"]), "ok": True,
                   "output": parse(row["path"]), "elapsed": round(time.time()-t, 2)}
            n_ok += 1
        except Exception as e:
            rec = {"file_name": row["file_name"], "seq": int(row["seq"]), "ok": False,
                   "output": "", "error": str(e)[:300], "elapsed": round(time.time()-t, 2)}
            n_err += 1
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        f.flush()
        n = n_ok + n_err
        if n % 25 == 0:
            el = time.time() - t0
            print(f"{n:>4}/{END}  경과 {el/60:.0f}분  남은 {(END-n)*el/n/60:.0f}분  실패 {n_err}")

print(f"\n완료 성공 {n_ok} 실패 {n_err} {(time.time()-t0)/60:.0f}분")
print(f"파일 {OUT.stat().st_size/1e6:.1f}MB / {sum(1 for _ in OUT.open(encoding='utf-8'))}건")

  25/150  경과 8분  남은 41분  실패 0
  50/150  경과 16분  남은 32분  실패 0
  75/150  경과 24분  남은 24분  실패 0
 100/150  경과 32분  남은 16분  실패 0
 125/150  경과 40분  남은 8분  실패 0
 150/150  경과 48분  남은 0분  실패 0

완료 성공 150 실패 0 48분
파일 0.2MB / 150건
